### TODO:
1. Load multiple dataset for different buildings.
2. Get the climate info for the state.
3. preprocess the data.
4. Create a linear regression model.
5. Prepare some tests for the model.

In [ ]:
import polars as pl
import pyarrow
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path
folder= Path().resolve().parent /'input'


### 1. Load multiple dataset for different buildings.

In [ ]:
# read the original data for three individual buildings
data=pl.scan_parquet(folder / 'load_profile_buildingID_*')

In [ ]:
# extract the buildings Ids
bldgs=data.select(pl.col('bldg_id').unique()).collect().to_series().to_list()

### 2. Get the climate info for the state.

In [ ]:
# extract the climate zone details based on the builings and cast the building id type to match the one in the original data
meta_data=(
    pl.scan_parquet(folder / 'MetaData.parquet')
)            
MetaData=meta_data.filter(
            pl.col('bldg_id').is_in(bldgs)).cast({"bldg_id": pl.UInt32}).select(
            pl.col('bldg_id', 'in.ashrae_iecc_climate_zone_2004_sub_cz_split'))

# MetaData.row(0, named=True) 
MetaData.collect()

In [ ]:
%%time
# merge the climate zone changes into the original data
df=data.join(MetaData, on='bldg_id')
df.collect_schema()

There are only time two climate zones for the data

----------------------------------------------

## Explore the data to get insights

In [ ]:
import polars.selectors as pls
df=df.select(pls.matches(r'bldg_id|timestamp|in.ashrae_iecc_climate_zone_*|out.electricity.*?.energy_consumption..kwh'))

In [ ]:
meta_data.collect()

## Preprocess the data

In [ ]:
y

In [108]:
# defining the independant and dependant variables with encoding categorical variables and converting the datetime to a timestamp
x= df.select(pl.col("timestamp").dt.timestamp(), pl.col('out.electricity.total.energy_consumption..kwh').alias('Total Usage'),
             pl.col('in.ashrae_iecc_climate_zone_2004_sub_cz_split').cast(pl.Categorical).cast(pl.UInt32).alias('Climate Zone')).collect()
y=df.select(pl.col("out.electricity.cooling.energy_consumption..kwh").alias('Cooling Consumption'),
           pl.col("out.electricity.heating.energy_consumption..kwh").alias('Heating Consumption'),
           pl.col("out.electricity.freezer.energy_consumption..kwh").alias('Freezing Consumption'),
           pl.col("out.electricity.refrigerator.energy_consumption..kwh").alias('Refrigerator Consumption'),
           pl.col("out.electricity.dishwasher.energy_consumption..kwh").alias('Dishwasher Consumption')).collect()

In [ ]:
# prepare measurement
x.select(pls.)

## test standardizing the data with the standard scaler

In [101]:
# from sklearn.preprocessing import StandardScaler
# scaler=StandardScaler()
# x=scaler.fit_transform(x)

## test standardizing the data with the Min Max Scaler

In [102]:
# from sklearn.preprocessing import MinMaxScaler
# scaler=MinMaxScaler()
# x=scaler.fit_transform(x)

### Prepare cross-validation model


In [106]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
def model(mod, x_train, x_test, y_train, y_test):
    mod.fit(x_train, y_train)
    predicted=mod.predict(x_test)
    # print(np.var(predicted))
    return mean_absolute_error(y_test, predicted)
        # r2_score(y_test, predicted)
        # mean_absolute_error(y_test, predicted), 
        # mean_squared_error(y_test, predicted), 
        # root_mean_squared_error(y_test, predicted)

In [107]:
%%time
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xg
skf= StratifiedKFold(n_splits=6)
kf=KFold(n_splits=10)
arr=[]
lf=pd.DataFrame()
for i, (train,test) in enumerate(kf.split(x,y)):
    x_train, x_test, y_train, y_test=x[train], x[test], y[train], y[test]
    LR=model(LinearRegression(), x_train, x_test, y_train, y_test)
    RF=model(RandomForestRegressor(n_estimators=3, max_depth=2, n_jobs=-1), x_train, x_test, y_train, y_test)
    XG=model(xg.XGBRegressor(), x_train, x_test, y_train, y_test)
    frame=pd.DataFrame({
    "Split Number": [i],
    "Linear Regression": [LR],
    "Random Forest": [RF],
    "XG Boost": [XG]
    })
    arr.append(frame)
lf=pd.concat(arr)

shape: (10_512, 3)
┌──────────────────┬─────────────┬──────────────┐
│ timestamp        ┆ Total Usage ┆ Climate Zone │
│ ---              ┆ ---         ┆ ---          │
│ i64              ┆ f32         ┆ u32          │
╞══════════════════╪═════════════╪══════════════╡
│ 1514765700000000 ┆ 0.1662      ┆ 0            │
│ 1514766600000000 ┆ 0.15641     ┆ 0            │
│ 1514767500000000 ┆ 0.14339     ┆ 0            │
│ 1514768400000000 ┆ 0.14249     ┆ 0            │
│ 1514769300000000 ┆ 0.14133     ┆ 0            │
│ …                ┆ …           ┆ …            │
│ 1524222000000000 ┆ 0.50114     ┆ 0            │
│ 1524222900000000 ┆ 1.52086     ┆ 0            │
│ 1524223800000000 ┆ 1.12717     ┆ 0            │
│ 1524224700000000 ┆ 0.33384     ┆ 0            │
│ 1524225600000000 ┆ 0.34802     ┆ 0            │
└──────────────────┴─────────────┴──────────────┘
shape: (10_512, 3)
┌──────────────────┬─────────────┬──────────────┐
│ timestamp        ┆ Total Usage ┆ Climate Zone │
│ ---       

## Calculating measures

In [105]:
lf.mean()

Split Number         4.500000
Linear Regression    0.028079
Random Forest        0.022351
XG Boost             0.019776
dtype: float64

## Observation

Apparently using r square as a metric to represent to the data is a misrepresentation to the model's performance.
R square formula is - >1- ( total of squared residulas /total of squared standard deviation)
Knowing MAE is the total of residuals -> total(y - yPredicted)
giving r square = 0.02 therefore
giving standard deviation of .001 as given above
then r square= .0004 / .00001= 40 -> 1-40 =-39 which is wrong

Therefore, MAE score is the perfect representation of the data.
another testing metric should be provided.